# 03 — Feature Engineering

This notebook creates derived features from the cleaned dataset.

**Key principle:** Every feature must be available at ORDER TIME.
We cannot use any information that only becomes available after delivery.

**Important:** Aggregated features (category averages, frequencies) are
computed from TRAINING data only, then mapped to test data. This prevents
data leakage through the feature engineering step.

In [1]:
import pandas as pd
import numpy as np
import sys
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

from src.features import (
    create_financial_features,
    create_shipping_features,
    create_aggregated_features
)
from sklearn.model_selection import train_test_split

## 3.1 Load Cleaned Data

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")
print(f"Loaded: {df.shape}")
df.head(3)

Loaded: (180519, 36)


,Type,Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,Customer State,...,Order Status,Product Name,Product Price,Product Status,Shipping Mode,Order_Year,Order_Month,Order_Day,Order_DayOfWeek,Order_Hour
0,DEBIT,4,91.250000,314.640015,0,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,...,COMPLETE,Smart watch,327.75,0,Standard Class,2018,1,31,Wednesday,22
1,TRANSFER,4,-249.089996,311.359985,1,Sporting Goods,Caguas,Puerto Rico,Consumer,PR,...,PENDING,Smart watch,327.75,0,Standard Class,2018,1,13,Saturday,12
2,CASH,4,-247.779999,309.720001,0,Sporting Goods,San Jose,EE. UU.,Consumer,CA,...,CLOSED,Smart watch,327.75,0,Standard Class,2018,1,13,Saturday,12


## 3.2 Train/Test Split — BEFORE Aggregated Features

We split the data FIRST so that aggregated features (like average sales per category)
are computed only from training data. This prevents a subtle form of data leakage.

If we computed these on the full dataset, the model would indirectly "see" information
from the test set during training.

In [3]:
X = df.drop("Late_delivery_risk", axis=1)
y = df["Late_delivery_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTarget distribution (train):")
print(y_train.value_counts(normalize=True).round(3))

Training set: (144415, 35)
Test set: (36104, 35)

Target distribution (train):
Late_delivery_risk
1    0.548
0    0.452
Name: proportion, dtype: float64


## 3.3 Non-Aggregated Features

These features are computed independently for each row, so there's no
leakage concern. We can apply them to train and test separately.

| Feature | Description | Available at order time? |
|---------|-------------|------------------------|
| `High_Value_Order` | 1 if sales > median | ✅ Yes — order value is known |
| `Discount_Percentage` | Discount as % of item value | ✅ Yes — discount is applied at order |
| `Sales_Per_Item` | Sales / quantity | ✅ Yes |
| `Profit_Margin` | Benefit / sales × 100 | ✅ Yes — expected profit margin |
| `Urgent_Shipment` | 1 if scheduled days ≤ 2 | ✅ Yes — shipping option is chosen at order |
| `Order_Size` | Quantity × price | ✅ Yes |
| `Price_Difference` | Product price − item product price | ✅ Yes |

In [4]:
X_train = create_financial_features(X_train)
X_train = create_shipping_features(X_train)

X_test = create_financial_features(X_test)
X_test = create_shipping_features(X_test)

print(f"\nTrain shape after features: {X_train.shape}")
print(f"Test shape after features: {X_test.shape}")

Created financial features: High_Value_Order, Discount_Percentage, Sales_Per_Item, Profit_Margin, Order_Size, Price_Difference
Created shipping features: Urgent_Shipment
Created financial features: High_Value_Order, Discount_Percentage, Sales_Per_Item, Profit_Margin, Order_Size, Price_Difference
Created shipping features: Urgent_Shipment

Train shape after features: (144415, 42)
Test shape after features: (36104, 42)


## 3.4 Aggregated Features (Leakage-Safe)

These features use information aggregated across multiple rows.
We compute them from TRAINING data only, then map to test.

| Feature | What it captures |
|---------|------------------|
| `Category_Avg_Sales` | Typical order value for this product category |
| `Category_Frequency` | How popular this category is (proxy for demand) |
| `Market_Frequency` | Order volume in this market (proxy for market size) |

In [5]:
X_train, X_test = create_aggregated_features(X_train, X_test)

print(f"\nFinal train shape: {X_train.shape}")
print(f"Final test shape: {X_test.shape}")

Created aggregated features (from training data): Category_Avg_Sales, Category_Frequency, Market_Frequency

Final train shape: (144415, 45)
Final test shape: (36104, 45)


## 3.5 Review Engineered Features

In [6]:
new_features = ["High_Value_Order", "Discount_Percentage", "Sales_Per_Item",
                "Profit_Margin", "Urgent_Shipment", "Order_Size",
                "Price_Difference", "Category_Avg_Sales",
                "Category_Frequency", "Market_Frequency"]

print("Engineered feature statistics (training set):")
X_train[new_features].describe().round(2).T

Engineered feature statistics (training set):


,count,mean,std,min,25%,50%,75%,max
High_Value_Order,144415.0,0.49,0.50,0.00,0.00,0.00,1.00,1.00
Discount_Percentage,144415.0,10.15,7.07,0.00,4.00,9.97,16.00,25.04
Sales_Per_Item,144415.0,141.08,139.44,9.99,50.00,59.99,199.99,1999.99
Profit_Margin,144415.0,10.85,42.05,-275.00,6.22,24.28,33.60,50.02
Urgent_Shipment,144415.0,0.40,0.49,0.00,0.00,0.00,1.00,1.00
Order_Size,144415.0,203.72,131.99,9.99,119.98,199.92,299.95,1999.99
Price_Difference,144415.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Category_Avg_Sales,144415.0,203.72,113.93,11.29,129.99,180.39,295.48,1500.00
Category_Frequency,144415.0,13063.83,5702.93,51.00,10025.00,13840.00,17747.00,19732.00
Market_Frequency,144415.0,34078.16,9531.72,9281.00,32932.00,40286.00,41229.00,41229.00


## 3.6 Identify Column Types for Preprocessing Pipeline

In [7]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols}")

Categorical columns (16): ['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Shipping Mode', 'Order_DayOfWeek']

Numerical columns (29): ['Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Customer Zipcode', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Price', 'Product Status', 'Order_Year', 'Order_Month', 'Order_Day', 'Order_Hour', 'High_Value_Order', 'Discount_Percentage', 'Sales_Per_Item', 'Profit_Margin', 'Order_Size', 'Price_Difference', 'Urgent_Shipment', 'Category_Avg_Sales', 'Category_Frequency', 'Market_Frequency']


## 3.7 Save Processed Data

In [8]:
# Save train/test splits with engineered features
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved processed train/test data to data/processed/")

Saved processed train/test data to data/processed/


## Summary

**Features created:** 10 engineered features across 3 categories
- Financial (6): value, discount, margin, price metrics
- Shipping (1): urgency flag
- Aggregated (3): category and market statistics — computed from training data only

**Data leakage prevention:**
- Train/test split done BEFORE aggregated features
- Aggregations computed from training set only
- All features use information available at order time

**Next step:** Statistical analysis of key relationships